# A Multigroup 1D Detector

In this example we will explore leveraging an adjoint formulation of the linear boltzmann equations for computing response functions in a desired region of interest (RoI).

This is a complete simulation transport example. Each aspect of this example can be broken as follows:
- [Prerequisites](#prerequisites)
- [Geometry](#Geometry)
    - [Mesh](#mesh)
    - [Material IDs](#material-ids)
    - [Volumetric Source](#volumetric-source)
- [Solver](#Solver)
    - [Angular Quadrature](#angular-quadrature)
    - [Linear Boltzmann Solver](#linear-boltzmann-solver)
    - [A Region of Interest](#a-region-of-interest)
- [Adjoint Solver](#adjoint-solver)
    - [Adjoint Source](#adjoint-source)
    - [Adjoint Options](#adjoint-options)
    - [Evaluate Resposne](#evaluate-response)
- [Solution](#solution) 

## Prerequisites

Before running this example, make sure that the **Python module of OpenSn** was installed.

### Converting and Running this Notebook from the Terminal
To run this notebook from the terminal, simply type:

`jupyter nbconvert --to python --execute VolSrc_1D.ipynb`.

To run this notebook in parallel (for example, using 4 processes), simply type:

`mpiexec -n 4 jupyter nbconvert --to python --execute VolSrc_1D.ipynb`.

In [ ]:
from mpi4py import MPI
size = MPI.COMM_WORLD.size
rank = MPI.COMM_WORLD.rank
barrier = MPI.COMM_WORLD.barrier

if rank == 0:
    print(f"Running the first LBS Adjoint example with {size} MPI processors.")

### Import Requirements

Import required classes and functions from the Python interface of OpenSn. Make sure that the path
to PyOpenSn is appended to Python's PATH.

In [ ]:
# assuming that the execute dir is the notebook dir
# this line is not necessary when PyOpenSn is installed using pip
sys.path.append("../../../..")

from pyopensn.mesh import OrthogonalMeshGenerator
from pyopensn.xs import MultiGroupXS
from pyopensn.source import PointSource, VolumetricSource
from pyopensn.aquad import GLCProductQuadrature2DXY
from pyopensn.solver import DiscreteOrdinatesProblem, SteadyStateSolver
from pyopensn.response import ResponseEvaluator
from pyopensn.fieldfunc import FieldFunctionInterpolationVolume
from pyopensn.logvol import RPPLogicalVolume
from pyopensn.context import UseColor, Finalize


import os
import sys
import math
import numpy as np

## Geometry
### Mesh
Here, we will use the in-house orthogonal mesh generator for a simple Cartesian grid.

We first create a list of nodes for spatial dimension in X. Here, we mesh the X direction to account for material interfaces in our problem. 

For this, we will create a function `getNodes()`. Given material boundaries and the number of cells within each boundary the function returns a python list of node values. 

In [ ]:
def getNodes(n, dn, ref=1):
    '''
    n: list of material interfaces
    dn: number of cells for each region
    ref(default=1): refine the number of cells by an integer multiple
    '''
    dn = [i * ref for i in dn]
    nodes = []
    for i in range(len(n[:-1])):
        edges = np.linspace(n[i],n[i+1],dn[i]+1)
        if len(nodes) == 0:
            nodes = edges
        else:
            nodes = [*nodes, *edges[1:]]
    return list(nodes)

x = [0.0, 2.0, 4.0, 6.0, 7.0, 10.73, 13.27, 17.0 ,18.0, 20.0]
dx = [2, 4, 2, 2, 8, 6, 8, 2, 1]
nodesx = getNodes(x, dx, ref=10)

### Orthogonal Mesh Generation
We use the `OrthogonalMeshGenerator` and pass the list of nodes per dimension. Here, we pass our lsit of nodes for the x direction creating a 1D geometry of side length 10 with 100 square cells. 

In [ ]:
meshgen = OrthogonalMeshGenerator(node_sets=[nodesx])
grid = meshgen.Execute()
grid.SetUniformBlockID(0)

### Material IDs
When using the in-house `OrthogonalMeshGenerator`, no material IDs are assigned. The user needs to assign material IDs to all cells. We will begin by assigning the background material ID with a value of 0.

In [ ]:
background = RPPLogicalVolume(infx=True, infy=True, infz=True)
grid.SetBlockIDFromLogicalVolume(background, 0, True)

Next we will assign material IDs for the source and detector. When assigning material IDs via logical volumes the IDs for each cell is overriden by the most recent assignment. That is to say, assigning a material ID of 1 to any region in our domain will override the previously set ID to those cells. 

In [ ]:
source = RPPLogicalVolume(
            infx=True,
            infy=True,
            zmin=2.0, zmax=4.0,
)
grid.SetBlockIDFromLogicalVolume(source, 1, True)

casing = RPPLogicalVolume(
            infx=True,
            infy=True,
            zmin=6.0, zmax=18.0,
)
grid.SetBlockIDFromLogicalVolume(casing, 2, True)

hdpe = RPPLogicalVolume(
            infx=True,
            infy=True,
            zmin=7.0, zmax=17.0,
)
grid.SetBlockIDFromLogicalVolume(hdpe, 3, True)

det = RPPLogicalVolume(
            infx=True,
            infy=True,
            zmin=10.73, zmax=13.27,
)
grid.SetBlockIDFromLogicalVolume(det, 4, True)

### Cross Sections
In this problem we are running a multigroup problem for the absoprtion reaction rate in a He-3 detector. This begins with importing our multigroup cross sections. In this case we will be using the WIMS69 group structure for 4 materials:
- Air
- Aluminum
- HDPE
- He-3

One should note, that although we have 4 cross sections, we have 5 material IDs. This is to account for the volumetric source having its own ID for which we apply a source definition. 

In [ ]:
num_groups = 69
xs_air = MultiGroupXS()
xs_air.LoadFromOpenSn("WIMS69/Air.cxs")

xs_alu = MultiGroupXS()
xs_alu.LoadFromOpenSn("WIMS69/Aluminum.cxs")

xs_hdpe = MultiGroupXS()
xs_hdpe.LoadFromOpenSn("WIMS69/HDPE.cxs")

xs_he3 = MultiGroupXS()
xs_he3.LoadFromOpenSn("WIMS69/Helium.cxs")

xsecs = [{"block_ids": [0], "xs": xs_air},
         {"block_ids": [1], "xs": xs_alu},
         {"block_ids": [2], "xs": xs_alu},
         {"block_ids": [3], "xs": xs_hdpe},
         {"block_ids": [4], "xs": xs_he3}]

### Volumetric Source

For the volumetric source we first define the source sprectrum. In this case we will assign a uniform spectrum in energy distributed along the source. Using `VolumetricSource` we assign this group-wise source definition to material ID 1. 

In [ ]:
num_groups = 69
groups = [1.00000E-11, 5.00000E-09, 1.00000E-08, 1.50000E-08, 2.00000E-08, 
          2.50000E-08, 3.00000E-08, 3.50000E-08, 4.20000E-08, 5.00000E-08, 
          5.80000E-08, 6.70000E-08, 8.00000E-08, 1.00000E-07, 1.40000E-07, 
          1.80000E-07, 2.20000E-07, 2.50000E-07, 2.80000E-07, 3.00000E-07, 
          3.20000E-07, 3.50000E-07, 4.00000E-07, 5.00000E-07, 6.25000E-07, 
          7.80000E-07, 8.50000E-07, 9.10000E-07, 9.50000E-07, 9.72000E-07, 
          9.96000E-07, 1.02000E-06, 1.04500E-06, 1.07100E-06, 1.09700E-06, 
          1.12300E-06, 1.15000E-06, 1.30000E-06, 1.50000E-06, 2.10000E-06, 
          2.60000E-06, 3.30000E-06, 4.00000E-06, 9.87700E-06, 1.59680E-05, 
          2.77000E-05, 4.80520E-05, 7.55014E-05, 1.48729E-04, 3.67263E-04, 
          9.06899E-04, 1.42510E-03, 2.23945E-03, 3.51910E-03, 5.53000E-03, 
          9.11800E-03, 1.50300E-02, 2.47800E-02, 4.08500E-02, 6.73400E-02, 
          1.11000E-01, 1.83000E-01, 3.02500E-01, 5.00000E-01, 8.21000E-01, 
          1.35300E+00, 2.23100E+00, 3.67900E+00, 6.06550E+00, 1.00000E+01]

dE = groups[-1] - groups[0]
group_wdth = np.flip(np.diff(groups))
dL = 2.0

src_g = (group_wdth / dE / dL).tolist()
vol_src = VolumetricSource(block_ids=[1], group_strength=src_g)

## Solver
### Angular Quadrature
Since we are solving a 1D problem we will create Gauss-Legendre Product Quadrature for a 1D slab which requires the total angles. In this case we will use 1024 angles creating a 1D angular quadrature in $\mu$.

In [ ]:
pquad = GLProductQuadrature1DSlab(2048)

### Linear Boltzmann Solver
#### Options for the Linear Boltzmann Problem (LBS)
In the LBS block, we provide
+ The number of energy groups
+ The groupsets (with 0-indexing);
    + the handle for the angular quadrature
    + the angle aggregation
    + the solver type
    + tolerances
+ Cross section map
+ Solver options

In [ ]:
phys = DiscreteOrdinatesProblem(
    mesh=grid,
    num_groups=1,
    groupsets=[
        {
            "groups_from_to": (0, 0),
            "angular_quadrature": pquad,
            "angle_aggregation_num_subsets": 1,
            "inner_linear_method": "petsc_gmres",
            "l_abs_tol": 1.0e-0,
            "l_max_its": 500,
            "gmres_restart_interval": 50
        }
    ],
    xs_map=xsecs,
    options={
        "scattering_order": 0,
        "spatial_discretization": "pwld",
        "boundary_conditions": [
            {"name": "zmin", "type": "vacuum"},
            {"name": "zmax", "type": "vacuum"}
        ],
        "volumetric_sources": [vol_src],
    },
)

### Putting the Linear Boltzmann Solver Together
We then create the physics solver, initialize it, and execute it.

In [ ]:
ss_solver = SteadyStateSolver(lbs_problem=phys)
ss_solver.Initialize()

Note: In this tutorial we have decided to execute the linear Boltzmann solver in addition to the adjoint solver to illustrate their duality. This is not necessary to execute the adjoint solver. 

In [ ]:
ss_solver.Execute()

### A Region of Interest

With the LBS solver executed, we now have solutions to the transport problem. For post processing this data into a quantitiy of interest, we first define a region of interest, a portion of the domain for which the solution is important. This region will be the same as our detector.

In [ ]:
roi_vol = RPPLogicalVolume(
            infx=True,
            infy=True,
            zmin=10.73, zmax=13.27,
)

### Computing a Quantity of Interest

Having defined a ROI, we now create a `FieldFunction`. In OpenSn we define a `FieldFunction` for the response we will like to calcualte. In this case we are looking for the total $He^3$ absorption reaction rate in our detector: 

$$
\text{QoI} = \sum^G_g \, \int_\mathcal{D} \, \int_{(4\pi)}  \, \sigma^g_{det}(\vec{r}) \psi^g(\vec{r},\vec\Omega) \, d^3r \, d\Omega = \sum^G_g \, \int_\mathcal{D} \, \sigma^g_{det}(\vec{r}) \phi^g(\vec{r}) \, d^3r
$$

To get our detector response $\sigma^g_{det}=\sigma^{He3,g}_{a}$ we use a function `getXSec`.

In [ ]:
def getXSec(file):
    f = open(file,"r")
    sigma = []
    val = 0
    lines = f.readlines()
    for l in lines:
        if len(l.split()) > 0 and \
            l.split()[0] ==  "SIGMA_A_BEGIN":
            val = 1 
            continue 
        elif len(l.split()) > 0 and \
            l.split()[0] ==  "SIGMA_A_END":
            val = 0 
            continue 
        if val:    
            line = l.split()
            sigma.append(float(line[1]))
    return sigma
sigma_det = getXSec("WIMS69/Helium.cxs")

Thus, in OpenSn we will generate a scalar field function using `GetScalarFieldFunctionList` with a `sum` over the RoI. At each group the solution $\phi^g(\vec{r})$ is multiplied by the detector response at that group $\sigma^{He3,g}_{a}$. 

In [ ]:
fflist = phys.GetScalarFieldFunctionList(only_scalar_flux=False)

fields = []
fwd_flux = []
fwd_resp = []
for g in range(num_groups):
    ffi = FieldFunctionInterpolationVolume()
    ffi.SetOperationType("sum")  # OP_SUM operation
    ffi.SetLogicalVolume(roi)
    ffi.AddFieldFunction(fflist[g][0])
    ffi.Initialize()
    ffi.Execute()
    value = ffi.GetValue()

    fwd_flux.append(value)
    fwd_resp.append(sigma_det[g]*value)
    fields.append(fflist[g][0])

With our field function defined, we can also export the multi-group scalar flux, $\phi^g$, to a .vtu file using ``ExportMultipleToVTK``.

In [ ]:
FFGrid = FieldFunctionGridBased
FFGrid.ExportMultipleToVTK(fields, "Flux/VolFlux_p")

It is additionally benefitial to write out our data to a text file. We will use information from these files to compare forward and adjoint solutions.

In [ ]:
f = open("volsrc.osn", "w")
f.write("Group : Flux : Response\n")
f.write("-----------------------\n")
for g in range(num_groups):
    f.write(str(g)+" : "+str(fwd_flux[g])+" : "+str(fwd_resp[g])+"\n")
f.write("Total Flux : "+str(np.sum(fwd_flux))+"\n")
f.write("Total Response : "+str(np.sum(fwd_resp)))
f.close()

## Adjoint Solver
### Adjoint Source

For solving the 1D transport problem via the Adjoint operator we first define our adjoint source $q^{\dagger}$ to be the detector volume, which we re-define as the `roi_vol` and apply a source strength equivalent to the abosrption cross section of our detector $\sigma^{He3,g}_a$. (By default, adjoint sources are isotoropically distributed in angle across the souce domain)

To do this we create a `ResponseFunction`. A response function is a function that folds against the fwd source distrubtion, taking the place of defining the group strength. This can be used to specifiy responses to specific energy bins, or this case, capture our detector response.

In [ ]:
def ResponseFunction(xyz, mat_id):
    response = getXSec("WIMS69/Helium.cxs")
    return response
response_func = VectorSpatialFunction(ResponseFunction)

adj_src = VolumetricSource(logical_volume=roi_vol,
                           func=response_func)

### Adjoint Options 
For executing the adjoint solver we reassign our solver options including,
+ A toggle for the adjoint 
+ Updating the volumetric sources

Note: Once the adjoint data has been save locally you do not need to run `Execute` to run the response evaluator.

In [ ]:
phys.SetOptions(
    adjoint = True,
    spatial_discretization = "pwld",
    volumetric_sources = [adj_src],
)
ss_solver.Execute()

### Evaluate Response

When evaluating response functions via the adjoint its efficiency derives from having access to the adjoint solution $\phi^{\dagger,g}$. For this, we export the adjoint flux moments to a .h5 file using `WriteFluxMoments`. The adjoint data is then saved to a location of our choosing.  

In [ ]:
data_dir = path+"/Data"
if not os.path.isdir(data_dir):
    os.makedirs(data_dir)
    
phys.WriteFluxMoments(data_dir+"/AdjFlux_p")

With the adjoint data in hand, we now create a `ResponseEvaluator`. 

We now supply two additional options to the `evaluator`, the `buffers` (adjoint data) and the `sources`. For computing the detector response we make use of `EvaluateResponse` which given a buffer name will compute an inner product of the adjoint flux $\phi^{\dagger,}$ against the forward source.    

In [ ]:
fwd_src = {'material': [{'block_id': 1, 'strength': src_g}]}

evaluator = ResponseEvaluator(lbs_problem = phys)
evaluator.SetOptions(
    buffers = [{
        'name': 'detector', 
        'file_prefixes': {'flux_moments': data_dir+'/AdjFlux_p'}
    }],
    sources = fwd_src
)

adj_resp = evaluator.EvaluateResponse("detector")

We export the adjoint flux solution to a .vtu using `ExportMultiToVTK`, the same as previously done for the "forward" flux. 

In [ ]:
fflist = phys.GetScalarFieldFunctionList(only_scalar_flux=False)
fields = []
for g in range(num_groups):
    fields.append(fflist[g][0])

FFGrid = FieldFunctionGridBased
FFGrid.ExportMultipleToVTK(fields, "Flux/AdjFlux_p")

## Solution

The resulting flux solution for the forward and adjoint problem is shown below: 

![Flux](images/Detector.png)

For the QoI both methods gave the same solution:

QoI = 0.0789612269312955

## Finalize (for Jupyter Notebook only)

In Python script mode, PyOpenSn automatically handles environment termination. However, this
automatic finalization does not occur when running in a Jupyter notebook, so explicit finalization
of the environment at the end of the notebook is required. Do not call the finalization in Python
script mode, or in console mode.

Note that PyOpenSn's finalization must be called before MPI's finalization.


In [ ]:
from IPython import get_ipython

def finalize_env():
    Finalize()
    MPI.Finalize()

ipython_instance = get_ipython()
if ipython_instance is not None:
    ipython_instance.events.register("post_execute", finalize_env)